##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [1]:
# Install KerasHub
!pip install --upgrade keras-hub keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.10.0
    Uninstalling keras-3.10.0:
      Successfully uninstalled keras-3.10.0
  Attempting uninstall: keras-hub
    Found existing installation: keras-hub 0.21.1
    Uninstalling keras-hub-0.21.1:
      Successfully uninstalled keras-hub-0.21.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.21.1 requires keras-hub==0.21.1, but you have keras-hub 0.26.0 which is incompatible.


In [10]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
import keras_hub
import requests
from PIL import Image
import io

# Load ImageNet Labels
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_labels = requests.get(labels_url).text.splitlines()

# REQUIRED KerasHub Components
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)
classifier = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet",
    preprocessor=preprocessor,
    activation="softmax"
)

# Verified Image URLs (Direct links to ImageNet classes)
image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/a/aa/California_quail.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/0/0f/Grosser_Panda.JPG",
    "https://upload.wikimedia.org/wikipedia/commons/b/bb/Kittyply_edit1.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/3/37/African_Bush_Elephant.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/2/25/Common_goldfish.JPG"
]

def get_valid_batch(urls):
    images = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    for url in urls:
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            images.append(np.array(img))
        except Exception as e:
            print(f"Skipping {url}: {e}")
    return images

# Predict
image_list = get_valid_batch(image_urls)

predictions = [classifier.predict(np.expand_dims(img, axis=0), verbose=0) for img in image_list]
results = [imagenet_labels[np.argmax(p)] for p in predictions]

print(f"\n{'Image Source':<20} | {'ViT Prediction'}")
print("-" * 40)
for url, pred in zip(image_urls, results):
    print(f"{url.split('/')[-1][:18]:<20} | {pred}")


Image Source         | ViT Prediction
----------------------------------------
California_quail.j   | quail
Grosser_Panda.JPG    | giant panda
Kittyply_edit1.jpg   | tabby
African_Bush_Eleph   | African elephant
Common_goldfish.JP   | goldfish


In [11]:
import pandas as pd

data = {
    "Image File": [
        "California_quail.jpg",
        "Grosser_Panda.JPG",
        "Kittyply_edit1.jpg",
        "African_Bush_Elephant.jpg",
        "Common_goldfish.JPG"
    ],
    "Predicted Label": [
        "quail",
        "giant panda",
        "tabby",
        "African elephant",
        "goldfish"
    ],
    "True Label (What you searched)": [
        "California Quail",
        "Giant Panda",
        "Tabby Cat",
        "African Elephant",
        "Goldfish"
    ],
    "Correct? (Yes/No)": ["Yes", "Yes", "Yes", "Yes", "Yes"]
}

df = pd.DataFrame(data)

print(df.to_string(index=False))



               Image File  Predicted Label True Label (What you searched) Correct? (Yes/No)
     California_quail.jpg            quail               California Quail               Yes
        Grosser_Panda.JPG      giant panda                    Giant Panda               Yes
       Kittyply_edit1.jpg            tabby                      Tabby Cat               Yes
African_Bush_Elephant.jpg African elephant               African Elephant               Yes
      Common_goldfish.JPG         goldfish                       Goldfish               Yes
